# Data Understanding

This notebook loads the raw `dataset_malwares.csv` (19,611 labelled Windows PE-header rows) and takes a first, uncleaned look at it: shape, columns, data types, class balance, missing values, duplicates, and feature variance. No cleaning happens here, only observation. Cleaning decisions based on what is found here are made and applied in `02_data_preparation.ipynb`.

In [1]:
# Standard library
import sys

# Third-party
import pandas as pd

# Show every column when previewing a DataFrame, instead of truncating with "..."
pd.set_option("display.max_columns", None)

# Make src/ importable from notebooks/
sys.path.append("../src")

from constants import ORDER_OF_FEATURES, ID_COLUMNS, LABEL_COLUMN, DROPPED_FEATURES, CORE_TRAITS

## Load the Dataset

In [2]:
# Load the raw dataset and report its shape
df = pd.read_csv("../data/dataset_malwares.csv")
print(df.shape)

(19611, 79)


The dataset contains 19,611 rows and 79 columns: `Name` (an identifier, not a feature), `Malware` (the label, 1 = malicious, 0 = benign), and 77 numeric PE-header features. `constants.py`'s `ORDER_OF_FEATURES` is the single source of truth for the 77 feature names and their order, so both this analysis and the deployed model always refer to the same columns.

## Preview the Data

In [3]:
df.head()

,Name,e_magic,e_cblp,e_cp,e_crlc,e_cparhdr,e_minalloc,e_maxalloc,e_ss,e_sp,e_csum,e_ip,e_cs,e_lfarlc,e_ovno,e_oemid,e_oeminfo,e_lfanew,Machine,NumberOfSections,TimeDateStamp,PointerToSymbolTable,NumberOfSymbols,SizeOfOptionalHeader,Characteristics,Magic,MajorLinkerVersion,MinorLinkerVersion,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,AddressOfEntryPoint,BaseOfCode,ImageBase,SectionAlignment,FileAlignment,MajorOperatingSystemVersion,MinorOperatingSystemVersion,MajorImageVersion,MinorImageVersion,MajorSubsystemVersion,MinorSubsystemVersion,SizeOfHeaders,CheckSum,SizeOfImage,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfStackCommit,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,NumberOfRvaAndSizes,Malware,SuspiciousImportFunctions,SuspiciousNameSection,SectionsLength,SectionMinEntropy,SectionMaxEntropy,SectionMinRawsize,SectionMaxRawsize,SectionMinVirtualsize,SectionMaxVirtualsize,SectionMaxPhysical,SectionMinPhysical,SectionMaxVirtual,SectionMinVirtual,SectionMaxPointerData,SectionMinPointerData,SectionMaxChar,SectionMainChar,DirectoryEntryImport,DirectoryEntryImportSize,DirectoryEntryExport,ImageDirectoryEntryExport,ImageDirectoryEntryImport,ImageDirectoryEntryResource,ImageDirectoryEntryException,ImageDirectoryEntrySecurity
0,VirusShare_a878ba26000edaac5c98eff4432723b3,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,248,34404,6,1236512358,0,0,240,34,523,8,0,54784,189440,0,51316,4096,4294967296,4096,512,6,0,6,0,5,2,1024,295281,274432,2,32832,524288,8192,1048576,4096,0,16,1,0,0,6,0.000000,0,512,0,274,0,188416,0,270336,0,245248,0,3758096608,0,7,152,0,0,54440,77824,73728,0
1,VirusShare_ef9130570fddc174b312b2047f5f4cf0,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,240,332,5,1365109591,0,0,224,258,267,9,0,205824,139264,0,84654,4096,4194304,4096,512,5,0,0,0,5,0,1024,0,442368,2,33088,1048576,4096,1048576,4096,0,16,1,0,0,5,3.815281,0,8704,0,24124,0,205680,0,339968,0,314880,0,3791650880,0,16,311,0,0,262276,294912,0,346112
2,VirusShare_ef84cdeba22be72a69b198213dada81a,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,256,332,6,1438777028,0,0,224,14,267,6,0,24576,20480,0,27364,256,4194304,4096,4096,4,0,0,0,4,0,4096,0,49152,2,0,1048576,4096,1048576,69632,0,528,1,0,0,6,0.103538,0,4096,0,329,0,24065,0,45056,0,45056,0,3221225536,0,6,176,0,0,36864,40960,0,0
3,VirusShare_6bf3608e60ebc16cbcff6ed5467d469e,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,128,332,7,1354629311,0,0,224,783,267,2,22,34304,28160,297472,16685,4096,4194304,4096,512,4,0,6,0,4,0,1024,14174816,1032192,2,32768,2097152,4096,1048576,4096,0,16,1,14,0,7,0.000000,0,0,0,144,0,638976,0,1003520,0,58880,0,3224371328,0,8,155,0,0,356352,1003520,0,14109472
4,VirusShare_2cc94d952b2efb13c7d6bbe0dd59d3fb,23117,144,3,0,4,0,65535,0,184,0,0,0,64,0,0,0,128,332,7,1386631250,0,0,224,783,267,2,56,8192,89600,512,4416,4096,4194304,4096,512,4,0,1,0,4,0,1024,0,110592,2,0,2097152,4096,1048576,4096,0,16,1,2,0,7,0.000000,0,0,0,24,0,42916,0,73728,0,54784,0,3227516992,0,2,43,0,0,61440,73728,0,90624


The 77 feature columns come from three raw structures inside every Windows PE file (the DOS header, the COFF/File header, and the Optional header), plus a set of derived features engineered on top of them (section statistics, suspicious import counts, and directory table addresses). `extract_features.py` is what actually computes these 77 numbers from a real `.exe`/`.dll` file; this dataset is a pre-extracted table of the same 77 numbers for 19,611 real samples.

## Statistical Summary of the Modelling Features

`describe()` earlier only summarised the 2 raw numeric columns pandas auto-detected as interesting; here it is run explicitly on `CORE_TRAITS`, the 10 features `constants.py` actually recommends for training (see `04_modelling_classical.ipynb`), since these are the columns whose distribution actually matters for modelling.

In [4]:
# Statistical summary of the 10 CORE_TRAITS columns actually used for modelling
df[CORE_TRAITS].describe()

,SuspiciousImportFunctions,SuspiciousNameSection,SectionsLength,SectionMinEntropy,SectionMinRawsize,SectionMinVirtualsize,SectionMaxPhysical,SectionMaxVirtual,SectionMaxPointerData,SectionMaxChar
count,19611.000000,19611.000000,19611.000000,19611.000000,1.961100e+04,1.961100e+04,1.961100e+04,1.961100e+04,1.961100e+04,1.961100e+04
mean,5.142675,0.018153,5.030340,1.547795,2.459628e+04,2.748368e+04,7.641715e+05,9.101676e+05,2.327726e+07,3.163632e+09
std,6.865930,0.183114,2.084558,1.810216,5.055981e+05,5.057358e+05,9.044052e+06,2.095545e+07,2.918776e+08,5.860332e+08
min,0.000000,0.000000,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+03,5.120000e+02,1.073742e+09
25%,0.000000,0.000000,4.000000,0.000000,0.000000e+00,3.240000e+02,5.342950e+04,7.782400e+04,4.556800e+04,3.221226e+09
50%,2.000000,0.000000,5.000000,0.402634,5.120000e+02,1.736000e+03,1.594600e+05,1.884160e+05,1.413120e+05,3.221226e+09
75%,8.000000,0.000000,6.000000,2.972017,4.096000e+03,6.578000e+03,3.901155e+05,5.406720e+05,4.057600e+05,3.221226e+09
max,45.000000,4.000000,79.000000,7.989859,4.884736e+07,4.884710e+07,8.302964e+08,2.684355e+09,3.763037e+09,4.294967e+09


Every `CORE_TRAITS` column is heavily right-skewed: the mean sits far above the median (50%) in every column, and the max is orders of magnitude above the 75th percentile (e.g. `SectionMinRawsize` has a median of 512 but a max of 48,847,360). This is expected for PE section sizes and pointers, most files have small, ordinary sections, and a handful have unusually large ones, and it is exactly the kind of long-tailed distribution that motivated using tree-based models (Random Forest, XGBoost) as the primary candidates in `04_modelling_classical.ipynb`: trees split on raw thresholds and are not sensitive to this kind of scale/skew the way a distance-based or linear model would be. `SuspiciousNameSection` stands out as the sparsest signal (75th percentile is still 0), consistent with it being a rare but strong indicator when it does appear, most files never disguise a section name, but the ones that do are worth flagging.

## Check Column Data Types

In [5]:
# Confirm every feature column is numeric; only Name is text
df.dtypes.value_counts()

int64      77
str         1
float64     1
Name: count, dtype: int64

All 77 feature columns are `int64` or `float64`. `Name` is the only `object` (text) column, and it is excluded from training via `ID_COLUMNS` in `constants.py`, it identifies a row, it does not describe the file's behaviour.

## Check the Class Balance

In [6]:
# Row counts and percentages for each label
print(df[LABEL_COLUMN].value_counts())
print(df[LABEL_COLUMN].value_counts(normalize=True) * 100)

Malware
1    14599
0     5012
Name: count, dtype: int64
Malware
1    74.442915
0    25.557085
Name: proportion, dtype: float64


74.44% of rows (14,599) are malicious and 25.56% (5,012) are benign, the opposite of the usual "malware is rare" intuition. The dataset is still imbalanced, just in the other direction, which is why every model trained on it later uses `class_weight="balanced"` (classical models) or a computed `scale_pos_weight` (XGBoost) rather than relying on accuracy alone.

## Inspect Missing Values

In [7]:
# Total missing values across the whole dataset
df.isna().sum().sum()

np.int64(0)

0 missing values anywhere in this dataset. The cleaning step in `02_data_preparation.ipynb` still checks for and drops missing values defensively, in case a future dataset update introduces gaps this one does not have.

## Check for Duplicate Records

In [8]:
# Full-row duplicates vs rows that share identical feature values (ignoring Name)
full_duplicates = df.duplicated().sum()
feature_duplicates = df.duplicated(subset=ORDER_OF_FEATURES).sum()
print("Full-row duplicates:", full_duplicates)
print("Duplicate feature-value rows (Name may differ):", feature_duplicates)

Full-row duplicates: 0
Duplicate feature-value rows (Name may differ): 2984


There are 0 full-row duplicates, but 2,984 rows share identical values across all 77 features with at least one other row (only `Name` differs). If left in, the exact same feature vector could land in both the training and test split, letting a model memorise that row instead of genuinely generalising. `02_data_preparation.ipynb` drops these.

## Check Feature Variance (Data Quality Check)

In [9]:
# A column with 0 variance holds the exact same value in every single row
variances = df[ORDER_OF_FEATURES].var()
zero_variance_columns = variances[variances == 0].index.tolist()
zero_variance_columns

['e_magic',
 'SectionMaxEntropy',
 'SectionMaxRawsize',
 'SectionMaxVirtualsize',
 'SectionMinPhysical',
 'SectionMinVirtual',
 'SectionMinPointerData',
 'SectionMainChar']

8 of the 77 feature columns have exactly 0 variance across all 19,611 rows. One, `e_magic`, is expected to be constant: it is the DOS header's "MZ" file-format signature, every valid PE file has the same value, so it is dropped separately in `RAW_HEADER_FEATURES` for carrying no information rather than being a data bug.

The other 7 (`SectionMaxEntropy`, `SectionMaxRawsize`, `SectionMaxVirtualsize`, `SectionMinPhysical`, `SectionMinVirtual`, `SectionMinPointerData`, `SectionMainChar`) describe things that genuinely vary between real files (entropy, section size), so a real file's version of these fields should never be constant. Finding them fixed at exactly one value across all 19,611 rows points to a construction bug in this dataset, not a real property of PE files. This is exactly why `constants.py` defines `DROPPED_FEATURES`, so a real file's genuine non-zero value at inference time is never mistaken for an anomaly by a model that only ever saw a constant during training.

In [10]:
# Confirm these 7 match constants.py's DROPPED_FEATURES exactly (e_magic handled separately)
set(zero_variance_columns) - {"e_magic"} == set(DROPPED_FEATURES)

True

## Check the DLL vs EXE Composition (a Look Ahead)

In [11]:
# Characteristics is a bitfield; 0x2000 is the IMAGE_FILE_DLL flag
IMAGE_FILE_DLL = 0x2000
is_dll = (df["Characteristics"].astype(int) & IMAGE_FILE_DLL) != 0
pd.crosstab(df[LABEL_COLUMN], is_dll, normalize="index") * 100

Characteristics,False,True
Malware,,
0,15.782123,84.217877
1,91.218577,8.781423


84.2% of benign rows are DLLs, but only 8.8% of malicious rows are (91.2% are EXEs or other non-DLL files). A rule as simple as "not a DLL -> malicious" would already score close to 89% accuracy with zero real learning about malware behaviour. This is investigated properly, with a train/test methodology built to catch it, in `03_eda.ipynb`, and is the reason the model's recommended feature set excludes raw header fields like `Characteristics` entirely (see `CORE_TRAITS` in `constants.py`).

## Summary

- Confirmed the dataset loads to 19,611 rows and 79 columns: `Name`, `Malware`, and 77 numeric PE-header features, all sourced from the same `ORDER_OF_FEATURES` used to extract features from real files.
- Confirmed all 77 feature columns are numeric; only `Name` is text.
- Ran `describe()` on the 10 `CORE_TRAITS` columns actually used for modelling: every one is heavily right-skewed (mean far above median, max orders of magnitude above the 75th percentile), motivating the choice of tree-based models in `04_modelling_classical.ipynb`, which split on raw thresholds and are not sensitive to this kind of scale/skew.
- Confirmed the class balance is 74.44% malicious versus 25.56% benign, motivating balanced class weighting in every model trained later.
- Confirmed there are 0 missing values in this dataset.
- Confirmed there are 0 full-row duplicates, but 2,984 rows that duplicate another row's feature values, which need to be dropped before splitting so the same row cannot appear in both train and test.
- Confirmed 8 columns have exactly 0 variance: `e_magic` (a constant PE signature, expected) and 7 others (`SectionMaxEntropy`, `SectionMaxRawsize`, `SectionMaxVirtualsize`, `SectionMinPhysical`, `SectionMinVirtual`, `SectionMinPointerData`, `SectionMainChar`) that should vary between real files but do not, a dataset construction bug behind `constants.py`'s `DROPPED_FEATURES`.
- Confirmed 84.2% of benign rows are DLLs versus only 8.8% of malicious rows, a strong shortcut a model could exploit instead of learning genuine malware behaviour, investigated further in `03_eda.ipynb`.
- No data was cleaned or transformed in this notebook. That begins in `02_data_preparation.ipynb`.